# Data Quality Report

**Project:** Mining EV Registration Patterns in Washington State  
**Data:** 50,000-record random sample (seed 42) from the Washington State EV Population dataset.

In [21]:
import pandas as pd

df = pd.read_csv(
    "data/ev_sample_50k.csv",
    sep="\t",
    low_memory=False
)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (50000, 16)


,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 GEOID
0,JTJABABB3R,Thurston,Lacey,WA,98516.0,2024,LEXUS,RZ,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,22.0,272203270,POINT (-122.78598 47.08476),PUGET SOUND ENERGY INC,5.306701e+10
1,7YAKPDDC6S,King,Seattle,WA,98103.0,2025,HYUNDAI,IONIQ 5,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,36.0,286868704,POINT (-122.34301 47.65918),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303300e+10
2,1V2WSPE85S,Clark,Vancouver,WA,98683.0,2025,VOLKSWAGEN,ID.4,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,17.0,285893233,POINT (-122.49212 45.60365),BONNEVILLE POWER ADMINISTRATION||PUD NO 1 OF C...,5.301104e+10
3,7PDSGABA6P,King,Sammamish,WA,98074.0,2023,RIVIAN,R1S,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,41.0,252551757,POINT (-122.03133 47.62858),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.303303e+10
4,7SAXCBE57R,Snohomish,Mill Creek,WA,98012.0,2024,TESLA,MODEL X,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,44.0,269998305,POINT (-122.20105 47.84423),PUGET SOUND ENERGY INC,5.306105e+10


## 1. Missing values

In [13]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing (%)": (df.isna().mean() * 100).round(3)
})

missing_summary = missing_summary.sort_values(
    "Missing Count", ascending=False
)

display(missing_summary)

print("Total missing cells:", df.isna().sum().sum())
print("Rows with missing values:", df.isna().any(axis=1).sum())

,Missing Count,Missing (%)
Legislative District,128,0.256
Electric Range,5,0.010
Vehicle Location,4,0.008
County,2,0.004
Electric Utility,2,0.004
City,2,0.004
Postal Code,2,0.004
2020 GEOID,2,0.004
Model,0,0.000
Make,0,0.000


Total missing cells: 147
Rows with missing values: 135


The sample has **147 missing cells across 135 records**. Legislative District has the most missing values (128). Zero electric ranges are examined separately.

## 2. Duplicates

In [14]:
duplicate_summary = pd.DataFrame({
    "Check": [
        "Exact duplicate rows",
        "Repeated DOL Vehicle IDs",
        "Missing DOL Vehicle IDs"
    ],
    "Count": [
        df.duplicated().sum(),
        df["DOL Vehicle ID"].duplicated().sum(),
        df["DOL Vehicle ID"].isna().sum()
    ]
})

display(duplicate_summary)

print("Unique DOL Vehicle IDs:", df["DOL Vehicle ID"].nunique())

,Check,Count
0,Exact duplicate rows,0
1,Repeated DOL Vehicle IDs,0
2,Missing DOL Vehicle IDs,0


Unique DOL Vehicle IDs: 50000


No exact duplicate rows or repeated or missing DOL Vehicle IDs were found.

## 3. Invalid and questionable values

In [15]:
year = pd.to_numeric(df["Model Year"], errors="coerce")
ev_range = pd.to_numeric(df["Electric Range"], errors="coerce")

invalid_summary = pd.DataFrame({
    "Check": [
        "Missing or nonnumeric model years",
        "Model years before 1990",
        "Model years after 2027",
        "Negative electric ranges",
        "Zero electric ranges",
        "Missing or nonnumeric electric ranges"
    ],
    "Count": [
        year.isna().sum(),
        (year < 1990).sum(),
        (year > 2027).sum(),
        (ev_range < 0).sum(),
        (ev_range == 0).sum(),
        ev_range.isna().sum()
    ]
})

display(invalid_summary)

,Check,Count
0,Missing or nonnumeric model years,0
1,Model years before 1990,0
2,Model years after 2027,0
3,Negative electric ranges,0
4,Zero electric ranges,32752
5,Missing or nonnumeric electric ranges,5


In [17]:
# Geographic and categorical validation

postal = df["Postal Code"].dropna().astype(str).str.replace(
    r"\.0$", "", regex=True
)

ev_types = [
    "Battery Electric Vehicle (BEV)",
    "Plug-in Hybrid Electric Vehicle (PHEV)"
]

validation_summary = pd.DataFrame({
    "Check": [
        "Invalid postal code format",
        "Unexpected EV types",
        "Missing state values",
        "Non-Washington addresses"
    ],
    "Count": [
        (~postal.str.fullmatch(r"\d{5}")).sum(),
        (~df["Electric Vehicle Type"].isin(ev_types)).sum(),
        df["State"].isna().sum(),
        df["State"].notna().sum() -
        (df["State"] == "WA").sum()
    ]
})

display(validation_summary)

,Check,Count
0,Invalid postal code format,5
1,Unexpected EV types,0
2,Missing state values,0
3,Non-Washington addresses,127


In [18]:
postal_clean = (
    df["Postal Code"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)

flagged_postal = df[
    postal_clean.notna() &
    ~postal_clean.str.fullmatch(r"\d{5}").fillna(False)
]

display(
    flagged_postal[
        ["State", "County", "City", "Postal Code", "DOL Vehicle ID"]
    ]
)

print("Flagged records:", len(flagged_postal))

,State,County,City,Postal Code,DOL Vehicle ID
3810,NH,Rockingham,Epping,3042.0,195337425
16128,NJ,Burlington,Joint Base Mdl,8641.0,151246350
18256,MA,Suffolk,Hyde Park,2136.0,154989264
22713,NJ,Morris,Mendham,7945.0,276510048
36843,CT,Fairfield,Greenwich,6830.0,225981486


Flagged records: 5


No negative electric ranges or model years outside the screening thresholds were found. **32,752** electric ranges are recorded as zero and **5** are missing. Five postal codes lost a leading zero when read as numbers; **127** records have non-Washington addresses. These are not automatically errors.

## 4. Outliers

In [19]:
# Electric range outliers by EV type

results = []

for ev_type, group in df.groupby("Electric Vehicle Type"):
    ranges = group.loc[
        group["Electric Range"] > 0, "Electric Range"
    ]

    q1 = ranges.quantile(0.25)
    q3 = ranges.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_count = (
        (ranges < lower) | (ranges > upper)
    ).sum()

    results.append({
        "EV Type": "BEV" if "(BEV)" in ev_type else "PHEV",
        "Valid Ranges": len(ranges),
        "Q1": q1,
        "Q3": q3,
        "Lower Bound": lower,
        "Upper Bound": upper,
        "Potential Outliers": outlier_count
    })

outlier_summary = pd.DataFrame(results)

display(outlier_summary)

,EV Type,Valid Ranges,Q1,Q3,Lower Bound,Upper Bound,Potential Outliers
0,BEV,7522,150.0,238.0,18.0,370.0,0
1,PHEV,9721,22.0,38.0,-2.0,62.0,174


Using the IQR rule separately on positive ranges for each vehicle type, **174 PHEVs** and **0 BEVs** were flagged. These are potential outliers, not confirmed data errors.

## 5. Cleaning decisions

In [20]:
cleaning_decisions = pd.DataFrame({
    "Issue": [
        "Missing Legislative District",
        "Missing Electric Range",
        "Zero Electric Range",
        "Missing geographic attributes",
        "Postal code formatting",
        "Non-Washington addresses",
        "Duplicate records",
        "Potential PHEV range outliers"
    ],
    "Recommended Action": [
        "Keep missing unless needed for analysis",
        "Treat as unknown",
        "Flag as unavailable range information",
        "Exclude only when required for geographic analysis",
        "Convert postal codes to five-digit strings",
        "Exclude from Washington-only clustering",
        "No removal needed",
        "Retain and investigate"
    ]
})

display(cleaning_decisions)

,Issue,Recommended Action
0,Missing Legislative District,Keep missing unless needed for analysis
1,Missing Electric Range,Treat as unknown
2,Zero Electric Range,Flag as unavailable range information
3,Missing geographic attributes,Exclude only when required for geographic anal...
4,Postal code formatting,Convert postal codes to five-digit strings
5,Non-Washington addresses,Exclude from Washington-only clustering
6,Duplicate records,No removal needed
7,Potential PHEV range outliers,Retain and investigate
